# Test pipeline feature (feature_day + feature + merge)

Notebook test từng bước của pipeline feature, chạy trực tiếp (không qua `run.py` CLI/Airflow). Tự đọc `features.yaml` nên thêm feature mới (như `age`) vào yaml là notebook tự chạy theo, không cần sửa notebook:
1. Cài dependency (nếu thiếu) + trỏ MinIO/Postgres về `localhost` (vì notebook chạy trên host, không nằm trong docker network `database_default` như khi chạy qua `docker exec`).
2. Xem config: `features.yaml` có `config` (áp dụng chung) + từng feature (`day_prefix`, `month_prefix`, `input`).
3. Ví dụ gọi trực tiếp 1 hàm giai đoạn ngày (`transform_order_by_day`) để thấy rõ tham số cần gì.
4. Test giai đoạn ngày cho **tất cả** `day_feature` khai báo trong `features.yaml`, qua dispatcher `run_feature_day`.
5. Test giai đoạn tháng cho **tất cả** rule instance khai báo trong `features.yaml`, qua dispatcher `run_feature`.
6. Ghép (merge) kết quả tất cả feature lại theo tháng, qua `merge_features_for_month`.
7. Đọc lại kết quả từng feature + kết quả đã ghép trên MinIO để kiểm tra.

Yêu cầu trước khi chạy trên local máy: `docker compose up -d` trong `docker/database/` và đã chạy `scripts/upload_raw_to_minio.py` để có raw data mẫu.

**Nếu gặp `ImportError: cannot import name '...'`** dù hàm đó đã có trong file `.py`: kernel đang cache module cũ từ lần import trước (khi code trong `src/` được sửa sau đó). Chạy cell "reload module" bên dưới (không cần restart kernel), rồi chạy lại cell bị lỗi.

In [1]:
# Cài dependency còn thiếu (bỏ qua nếu đã có)
%pip install -q pandas pyarrow "boto3==1.42.90" "botocore==1.42.90" s3fs pyyaml loguru

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import pathlib

# Thêm root repo vào sys.path để import được package src.*
REPO_ROOT = pathlib.Path.cwd().parent.parent
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: d:\vtp-template\vtp


In [3]:
# Reload module: chạy lại cell này (KHÔNG cần restart kernel) mỗi khi vừa
# sửa code trong src/ và gặp ImportError/hành vi cũ dù file trên đĩa đã đổi.
# Python cache module trong sys.modules — `from src.x import y` không đọc
# lại file nếu module `src.x` đã được import trước đó trong cùng kernel.
import sys

_removed = [m for m in list(sys.modules) if m.startswith("src.")]
for m in _removed:
    del sys.modules[m]
print(f"Đã xóa cache {len(_removed)} module 'src.*'")

Đã xóa cache 0 module 'src.*'


In [4]:
# connect.yaml khai báo host "minio"/"postgres".
from src.utils import minio_client

if not getattr(minio_client, "_patched_for_localhost", False):
    _original_get_minio_config = minio_client.get_minio_config

    def _get_minio_config_localhost():
        bucket_name, minio_endpoint, access_key, secret_key = _original_get_minio_config()
        minio_endpoint = minio_endpoint.replace("minio:", "localhost:")
        return bucket_name, minio_endpoint, access_key, secret_key

    minio_client.get_minio_config = _get_minio_config_localhost
    minio_client._patched_for_localhost = True

print(minio_client.get_minio_config())

('clean-202605', 'localhost:9990', 'minio', 'minio123')


## 0. Xem config trong `features.yaml`

`config` áp dụng chung cho mọi feature (đường dẫn raw, tên cột partition, nơi lưu kết quả merge). Mỗi feature khai báo riêng `day_feature`, `day_prefix`/`month_prefix`, `input` — không có hằng số nào hardcode trong code Python, `run.py` tự gộp hết lại rồi gọi hàm.

In [5]:
from src.utils.common import load_config

features_cfg = load_config("features.yaml")
global_cfg = features_cfg["config"]

print("config chung:", global_cfg)
print()
for f in features_cfg["features"]:
    print(f["name"], "->", f["day_feature"], "/", f["function"])
    print("   day_prefix:", f["day_prefix"], "| month_prefix:", f["month_prefix"])
    print("   input:", f.get("input"))

config chung: {'raw_day_prefix': 'raw/hitech_day', 'day_partition_key': 'partition_date', 'month_partition_key': 'partition', 'merged_prefix': 'clean/merged'}

order_l3m -> order / transform_order_lxm
   day_prefix: clean/fts/order/day | month_prefix: clean/fts/order/month_l3m
   input: {'months_window': 3, 'min_count': 10, 'min_value': 10000000}
order_l6m -> order / transform_order_lxm
   day_prefix: clean/fts/order/day | month_prefix: clean/fts/order/month_l6m
   input: {'months_window': 6, 'min_count': 10, 'min_value': 10000000}
age_l12m -> age / transform_age_lxm
   day_prefix: clean/fts/age/day | month_prefix: clean/fts/age/month_l12m
   input: {'months_window': 12, 'min_age': 20, 'max_age': 50}


## 1. Ví dụ gọi trực tiếp 1 hàm giai đoạn ngày

Lấy `order` làm ví dụ: `transform_order_by_day` nhận `day_prefix`/`raw_day_prefix`/`day_partition_key` qua tham số (không tự đọc config) — lấy từ `features.yaml` ra rồi truyền vào, để thấy đúng những gì `run_feature_day` làm hộ ở bước tiếp theo.

In [6]:
from src.data_processing.transform.order import transform_order_by_day

order_cfg = next(
    f for f in features_cfg["features"] if f["day_feature"] == "order"
)

day_df = transform_order_by_day(
    "20251201",
    day_prefix=order_cfg["day_prefix"],
    raw_day_prefix=global_cfg["raw_day_prefix"],
    day_partition_key=global_cfg["day_partition_key"],
)
print(day_df.shape)
day_df.head()

Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251201/data.parquet
(50, 3)


,cus_id,count,value
0,"{'member0': 4110000.0, 'member1': None}",0.0,941128
1,"{'member0': 4110001.0, 'member1': None}",0.0,617451
2,"{'member0': 4110002.0, 'member1': None}",0.0,2568848
3,"{'member0': 4110003.0, 'member1': None}",0.0,70058
4,"{'member0': 4110004.0, 'member1': None}",6.0,2212545


## 2. Giai đoạn ngày — tất cả `day_feature`, qua dispatcher `run_feature_day`

Đây là đường mà DAG/CLI thực tế dùng: tra `features.yaml` theo `day_feature`, tự gộp `config` chung + `day_prefix` rồi gọi hàm ngày tương ứng cho từng ngày — không cần tự truyền tham số như cell trên. Chạy cho **mọi** `day_feature` đang khai báo (hiện tại: `order`, `age`).

In [7]:
from src.data_processing.run import run_feature_day

DT_FROM, DT_TO = "20251201", "20251231"

day_features = sorted({f["day_feature"] for f in features_cfg["features"]})
print("day_features:", day_features)

for day_feature in day_features:
    run_feature_day(day_feature, DT_FROM, DT_TO)

2026-07-03 01:12:55.631 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251201
2026-07-03 01:12:55.730 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251202


day_features: ['age', 'order']
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/age/day/partition_date=20251201/data.parquet


2026-07-03 01:12:55.849 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251203
2026-07-03 01:12:55.981 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251204


Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/age/day/partition_date=20251202/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/age/day/partition_date=20251203/data.parquet


2026-07-03 01:12:56.099 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251205
2026-07-03 01:12:56.228 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'age' for date=20251206


...

Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251224/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251225/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251226/data.parquet


2026-07-03 01:13:01.632 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'order' for date=20251228
2026-07-03 01:13:01.724 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'order' for date=20251229


Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251227/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251228/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251229/data.parquet


2026-07-03 01:13:01.808 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'order' for date=20251230
2026-07-03 01:13:01.890 | INFO     | src.data_processing.run:run_feature_day:79 - Start day-feature 'order' for date=20251231


Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251230/data.parquet
Đã upload thành công 50 records lên MinIO: clean-202605/clean/fts/order/day/partition_date=20251231/data.parquet


## 3. Giai đoạn tháng (có window) — tất cả rule instance, qua dispatcher `run_feature`

`run_feature` tra config rồi gọi hàm tháng tương ứng (ví dụ `transform_order_lxm`, `transform_age_lxm`) — hàm tự load N tháng dữ liệu ngày đã clean ở bước trên, tính toán + so ngưỡng, tự lưu. Toàn bộ tham số lấy từ `features.yaml`. Chạy cho **mọi** rule instance (hiện tại: `order_l3m`, `order_l6m`, `age_l12m`).

In [8]:
from src.data_processing.run import run_feature

MONTH = "202512"

feature_names = [f["name"] for f in features_cfg["features"]]
print("feature_names:", feature_names)

for feature_name in feature_names:
    run_feature(feature_name, MONTH)

2026-07-03 01:13:01.994 | INFO     | src.data_processing.run:run_feature:101 - Start feature 'order_l3m' for month=202512


feature_names: ['order_l3m', 'order_l6m', 'age_l12m']


2026-07-03 01:13:02.286 | INFO     | src.data_processing.run:run_feature:103 - Feature 'order_l3m' completed
2026-07-03 01:13:02.296 | INFO     | src.data_processing.run:run_feature:101 - Start feature 'order_l6m' for month=202512


Đã upload thành công 1550 records lên MinIO: clean-202605/clean/fts/order/month_l3m/partition=202512/data.parquet


2026-07-03 01:13:02.522 | INFO     | src.data_processing.run:run_feature:103 - Feature 'order_l6m' completed
2026-07-03 01:13:02.535 | INFO     | src.data_processing.run:run_feature:101 - Start feature 'age_l12m' for month=202512


Đã upload thành công 1550 records lên MinIO: clean-202605/clean/fts/order/month_l6m/partition=202512/data.parquet


2026-07-03 01:13:02.808 | INFO     | src.data_processing.run:run_feature:103 - Feature 'age_l12m' completed


Đã upload thành công 1550 records lên MinIO: clean-202605/clean/fts/age/month_l12m/partition=202512/data.parquet


## 4. Ghép (merge) kết quả tất cả feature lại theo tháng

`merge_features_for_month(month)` tự đọc lại output giai đoạn tháng của **mọi** feature trong `features.yaml`, ghép theo `cus_id` (outer join), lưu xuống `merged_prefix`/`month_partition_key`={month}/. Đây là bước cuối trong DAG (chạy sau `run_feature`, gọi qua `run_merge_range(dt_from, dt_to)` để tự tìm đúng tháng như giai đoạn tháng).

In [9]:
from src.data_processing.run import merge_features_for_month

merged_df = merge_features_for_month(MONTH)
print(merged_df.shape)
merged_df.head()

Đã upload thành công 1550 records lên MinIO: clean-202605/clean/merged/partition=202512/data.parquet
(1550, 11)


,cus_id,f_order_avg_count_l3m,f_order_avg_value_l3m,f_order_count_ok_l3m,f_order_value_ok_l3m,f_order_avg_count_l6m,f_order_avg_value_l6m,f_order_count_ok_l6m,f_order_value_ok_l6m,f_age_l12m,f_age_ok_l12m
0,"{'member0': 4110000.0, 'member1': None}",0.0,941128.0,0,0,0.0,941128.0,0,0,48.925394,1
1,"{'member0': 4110001.0, 'member1': None}",0.0,617451.0,0,0,0.0,617451.0,0,0,NaN,0
2,"{'member0': 4110002.0, 'member1': None}",0.0,2568848.0,0,0,0.0,2568848.0,0,0,NaN,0
3,"{'member0': 4110003.0, 'member1': None}",0.0,70058.0,0,0,0.0,70058.0,0,0,NaN,0
4,"{'member0': 4110004.0, 'member1': None}",6.0,2212545.0,0,0,6.0,2212545.0,0,0,NaN,0


## 5. Đọc lại kết quả (từng feature + đã ghép) đã lưu trên MinIO để kiểm tra

In [10]:
import pandas as pd
from src.utils.minio_client import get_minio_config


def read_minio_parquet(prefix: str) -> pd.DataFrame:
    bucket_name, minio_endpoint, access_key, secret_key = get_minio_config()
    return pd.read_parquet(
        f"s3://{bucket_name}/{prefix}",
        storage_options={
            "endpoint_url": f"http://{minio_endpoint}",
            "key": access_key,
            "secret": secret_key,
        },
    )


for feature_cfg in features_cfg["features"]:
    prefix = f"{feature_cfg['month_prefix']}/{global_cfg['month_partition_key']}={MONTH}/"
    df = read_minio_parquet(prefix)
    print(feature_cfg["name"], "->", df.shape, list(df.columns))
    display(df.head())

merged_prefix = f"{global_cfg['merged_prefix']}/{global_cfg['month_partition_key']}={MONTH}/"
merged_check_df = read_minio_parquet(merged_prefix)
print("merged", "->", merged_check_df.shape, list(merged_check_df.columns))
display(merged_check_df.head())

order_l3m -> (1550, 5) ['cus_id', 'f_order_avg_count_l3m', 'f_order_avg_value_l3m', 'f_order_count_ok_l3m', 'f_order_value_ok_l3m']


,cus_id,f_order_avg_count_l3m,f_order_avg_value_l3m,f_order_count_ok_l3m,f_order_value_ok_l3m
0,"{'member0': 4110000.0, 'member1': None}",0.0,941128.0,0,0
1,"{'member0': 4110001.0, 'member1': None}",0.0,617451.0,0,0
2,"{'member0': 4110002.0, 'member1': None}",0.0,2568848.0,0,0
3,"{'member0': 4110003.0, 'member1': None}",0.0,70058.0,0,0
4,"{'member0': 4110004.0, 'member1': None}",6.0,2212545.0,0,0


order_l6m -> (1550, 5) ['cus_id', 'f_order_avg_count_l6m', 'f_order_avg_value_l6m', 'f_order_count_ok_l6m', 'f_order_value_ok_l6m']


,cus_id,f_order_avg_count_l6m,f_order_avg_value_l6m,f_order_count_ok_l6m,f_order_value_ok_l6m
0,"{'member0': 4110000.0, 'member1': None}",0.0,941128.0,0,0
1,"{'member0': 4110001.0, 'member1': None}",0.0,617451.0,0,0
2,"{'member0': 4110002.0, 'member1': None}",0.0,2568848.0,0,0
3,"{'member0': 4110003.0, 'member1': None}",0.0,70058.0,0,0
4,"{'member0': 4110004.0, 'member1': None}",6.0,2212545.0,0,0


age_l12m -> (1550, 3) ['cus_id', 'f_age_l12m', 'f_age_ok_l12m']


,cus_id,f_age_l12m,f_age_ok_l12m
0,"{'member0': 4110000.0, 'member1': None}",48.925394,1
1,"{'member0': 4110001.0, 'member1': None}",NaN,0
2,"{'member0': 4110002.0, 'member1': None}",NaN,0
3,"{'member0': 4110003.0, 'member1': None}",NaN,0
4,"{'member0': 4110004.0, 'member1': None}",NaN,0


merged -> (1550, 11) ['cus_id', 'f_order_avg_count_l3m', 'f_order_avg_value_l3m', 'f_order_count_ok_l3m', 'f_order_value_ok_l3m', 'f_order_avg_count_l6m', 'f_order_avg_value_l6m', 'f_order_count_ok_l6m', 'f_order_value_ok_l6m', 'f_age_l12m', 'f_age_ok_l12m']


,cus_id,f_order_avg_count_l3m,f_order_avg_value_l3m,f_order_count_ok_l3m,f_order_value_ok_l3m,f_order_avg_count_l6m,f_order_avg_value_l6m,f_order_count_ok_l6m,f_order_value_ok_l6m,f_age_l12m,f_age_ok_l12m
0,"{'member0': 4110000.0, 'member1': None}",0.0,941128.0,0,0,0.0,941128.0,0,0,48.925394,1
1,"{'member0': 4110001.0, 'member1': None}",0.0,617451.0,0,0,0.0,617451.0,0,0,NaN,0
2,"{'member0': 4110002.0, 'member1': None}",0.0,2568848.0,0,0,0.0,2568848.0,0,0,NaN,0
3,"{'member0': 4110003.0, 'member1': None}",0.0,70058.0,0,0,0.0,70058.0,0,0,NaN,0
4,"{'member0': 4110004.0, 'member1': None}",6.0,2212545.0,0,0,6.0,2212545.0,0,0,NaN,0


**Lưu ý:** data mẫu (`data/output/hitech_day`) là data fake:
- `cus_id` không lặp lại giữa các ngày, nên `order` avg 3m/6m mỗi khách hàng chỉ có 1 điểm dữ liệu, và cột `f_order_count_ok_l{N}m`/`f_order_value_ok_l{N}m` sẽ toàn 0 (chưa đủ dữ liệu cả `months_window` tháng).
- `age` không có tính chất "đủ tháng" như trên (chỉ lấy tuổi lớn nhất trong window) nên `f_age_ok_l12m` có cả 0 và 1, phản ánh đúng ngưỡng [min_age, max_age].
- Kết quả `merged` là outer-join theo `cus_id` của tất cả feature ở trên — số cột = tổng số cột output (trừ `cus_id` lặp) của từng feature.

Đây là đặc điểm của data mẫu, không phải lỗi pipeline.